In [9]:
import pandas as pd
import timeit

-  Здійснити data cleaning
-  Окремими функціями сформувати вибірки:
   
   
   


In [10]:
df=pd.read_csv("household_power_consumption.txt", sep=";", low_memory=False)
df.replace("?", pd.NA, inplace=True)
df.dropna(inplace=True)

def more_than_5kw(df):
    return df[df["Global_active_power"].astype(float) > 5.0]

def current_between_19_and_20(df):
    df_current = df[(df["Global_intensity"].astype(float).between(19, 20))]
    df_current = df_current[(df_current["Sub_metering_2"].astype(float) > (df_current["Sub_metering_1"].astype(float) + df_current["Sub_metering_3"].astype(float)))]  
    return df_current

def random_sample(df):
    df_sample = df.sample(n=500000, random_state=42, replace=False)  

    mean_sub_metering_1 = df_sample["Sub_metering_1"].astype(float).mean()
    mean_sub_metering_2 = df_sample["Sub_metering_2"].astype(float).mean()
    mean_sub_metering_3 = df_sample["Sub_metering_3"].astype(float).mean()

    print("Mean Sub_metering_1:", mean_sub_metering_1)
    print("Mean Sub_metering_2:", mean_sub_metering_2)
    print("Mean Sub_metering_3:", mean_sub_metering_3)
    return df_sample

def after_1800(df):
    df_after_1800 = df[df["Time"].str.startswith(("18:", "19:", "20:", "21:", "22:", "23:"))]
    df_after_1800 = df_after_1800[df_after_1800["Global_active_power"].astype(float) > 6.0]
    
    group_1 = df_after_1800["Sub_metering_1"].astype(float)
    group_2 = df_after_1800["Sub_metering_2"].astype(float)
    group_3 = df_after_1800["Sub_metering_3"].astype(float)

    df_evening_group2 = df_after_1800[(group_2 > group_1) & (group_2 > group_3)]

    half = len(df_evening_group2) // 2
    first_half = df_evening_group2.iloc[:half]
    second_half = df_evening_group2.iloc[half:]

    result = pd.concat([first_half.iloc[::3],second_half.iloc[::4]])

    return result






- Обрати всі записи, у яких загальна активна споживана потужність перевищує 5 кВт.

In [11]:
print(more_than_5kw(df).head())

          Date      Time Global_active_power Global_reactive_power  Voltage  \
1   16/12/2006  17:25:00               5.360                 0.436  233.630   
2   16/12/2006  17:26:00               5.374                 0.498  233.290   
3   16/12/2006  17:27:00               5.388                 0.502  233.740   
11  16/12/2006  17:35:00               5.412                 0.470  232.780   
12  16/12/2006  17:36:00               5.224                 0.478  232.990   

   Global_intensity Sub_metering_1 Sub_metering_2  Sub_metering_3  
1            23.000          0.000          1.000            16.0  
2            23.000          0.000          2.000            17.0  
3            23.000          0.000          1.000            17.0  
11           23.200          0.000          1.000            17.0  
12           22.400          0.000          1.000            16.0  


- Обрати всі записи, у яких сила струму лежить в межах 19-20 А, для них виявити ті, у яких пральна машина та холодильних споживають більше, ніж бойлер та кондиціонер.

In [12]:
print(current_between_19_and_20(df).head())

           Date      Time Global_active_power Global_reactive_power  Voltage  \
45   16/12/2006  18:09:00               4.464                 0.136  234.660   
460  17/12/2006  01:04:00               4.582                 0.258  238.080   
464  17/12/2006  01:08:00               4.618                 0.104  239.610   
475  17/12/2006  01:19:00               4.636                 0.140  237.370   
476  17/12/2006  01:20:00               4.634                 0.152  237.170   

    Global_intensity Sub_metering_1 Sub_metering_2  Sub_metering_3  
45            19.000          0.000         37.000            16.0  
460           19.600          0.000         13.000             0.0  
464           19.600          0.000         27.000             0.0  
475           19.400          0.000         36.000             0.0  
476           19.400          0.000         35.000             0.0  


- Обрати випадковим чином 500000 записів (без повторів елементів вибірки), для них обчислити середні величини усіх 3-х груп споживання електричної енергії

In [13]:
print(random_sample(df).head())

Mean Sub_metering_1: 1.119258
Mean Sub_metering_2: 1.308912
Mean Sub_metering_3: 6.45295
               Date      Time Global_active_power Global_reactive_power  \
1030580   1/12/2008  09:44:00               1.502                 0.074   
1815     17/12/2006  23:39:00               0.374                 0.264   
1295977    3/6/2009  17:01:00               0.620                 0.300   
206669     9/5/2007  05:53:00               0.280                 0.200   
1048893  14/12/2008  02:57:00               1.372                 0.054   

         Voltage Global_intensity Sub_metering_1 Sub_metering_2  \
1030580  240.170            6.400          0.000          0.000   
1815     245.500            1.800          0.000          2.000   
1295977  239.850            3.000          0.000          1.000   
206669   235.720            1.400          0.000          0.000   
1048893  243.950            5.600          0.000          0.000   

         Sub_metering_3  
1030580            18.0  
1815 

- Обрати ті записи, які після 18-00 споживають понад 6 кВт за хвилину в середньому, серед відібраних визначити ті, у яких основне споживання електроенергії у вказаний проміжок часу припадає на пральну машину, сушарку, холодильник та освітлення (група 2 є найбільшою), а потім обрати кожен третій результат із першої половини та кожен четвертий результат із другої половини.

In [14]:
print(after_1800(df).head())

             Date      Time Global_active_power Global_reactive_power  \
41     16/12/2006  18:05:00               6.052                 0.192   
44     16/12/2006  18:08:00               6.308                 0.116   
17494  28/12/2006  20:58:00               6.386                 0.374   
17498  28/12/2006  21:02:00               8.088                 0.262   
17501  28/12/2006  21:05:00               7.230                 0.152   

       Voltage Global_intensity Sub_metering_1 Sub_metering_2  Sub_metering_3  
41     232.930           26.200          0.000         37.000            17.0  
44     232.250           27.000          0.000         36.000            17.0  
17494  236.630           27.000          1.000         36.000            17.0  
17498  235.500           34.400          1.000         72.000            17.0  
17501  235.220           30.600          1.000         73.000            17.0  


- Пронормувати та стандартизувати вибраний датасет
- Підрахувати коефіцієнт Пірсона та Спірмена для двох integer/real атрибутів.
- Провести One Hot Encoding категоріального атрибута.

In [15]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler

numeric_cols = ["Global_active_power", "Global_reactive_power", "Voltage", "Global_intensity", 
                "Sub_metering_1", "Sub_metering_2", "Sub_metering_3"]

df[numeric_cols] = df[numeric_cols].astype(float)

minmax_scaler = MinMaxScaler()
df_normalized = pd.DataFrame(minmax_scaler.fit_transform(df[numeric_cols]), columns=[col + "_norm" for col in numeric_cols])

standard_scaler = StandardScaler()
df_standardized = pd.DataFrame(standard_scaler.fit_transform(df[numeric_cols]), columns=[col + "_std" for col in numeric_cols])

pearson_corr = df["Global_active_power"].corr(df["Voltage"], method="pearson")
spearman_corr = df["Global_active_power"].corr(df["Voltage"], method="spearman")
print("Pearson correlation (Global_active_power & Voltage):", pearson_corr)
print("Spearman correlation (Global_active_power & Voltage):", spearman_corr)

df_encoded = pd.get_dummies(df, columns=["Date", "Time"], drop_first=True)


Pearson correlation (Global_active_power & Voltage): -0.3997616096289585
Spearman correlation (Global_active_power & Voltage): -0.32521294259808614


## Normalized:

In [16]:
print(df_normalized.head())

   Global_active_power_norm  Global_reactive_power_norm  Voltage_norm  \
0                  0.374796                    0.300719      0.376090   
1                  0.478363                    0.313669      0.336995   
2                  0.479631                    0.358273      0.326010   
3                  0.480898                    0.361151      0.340549   
4                  0.325005                    0.379856      0.403231   

   Global_intensity_norm  Sub_metering_1_norm  Sub_metering_2_norm  \
0               0.377593                  0.0               0.0125   
1               0.473029                  0.0               0.0125   
2               0.473029                  0.0               0.0250   
3               0.473029                  0.0               0.0125   
4               0.323651                  0.0               0.0125   

   Sub_metering_3_norm  
0             0.548387  
1             0.516129  
2             0.548387  
3             0.548387  
4             0

## Standardized:

In [17]:
print(df_standardized.head())

   Global_active_power_std  Global_reactive_power_std  Voltage_std  \
0                 2.955077                   2.610721    -1.851816   
1                 4.037085                   2.770406    -2.225274   
2                 4.050326                   3.320432    -2.330213   
3                 4.063567                   3.355917    -2.191324   
4                 2.434881                   3.586573    -1.592556   

   Global_intensity_std  Sub_metering_1_std  Sub_metering_2_std  \
0              3.098789           -0.182337           -0.051274   
1              4.133800           -0.182337           -0.051274   
2              4.133800           -0.182337            0.120487   
3              4.133800           -0.182337           -0.051274   
4              2.513782           -0.182337           -0.051274   

   Sub_metering_3_std  
0            1.249421  
1            1.130897  
2            1.249421  
3            1.249421  
4            1.249421  


## Encoded:

In [18]:
print(df_encoded.head())

   Global_active_power  Global_reactive_power  Voltage  Global_intensity  \
0                4.216                  0.418   234.84              18.4   
1                5.360                  0.436   233.63              23.0   
2                5.374                  0.498   233.29              23.0   
3                5.388                  0.502   233.74              23.0   
4                3.666                  0.528   235.68              15.8   

   Sub_metering_1  Sub_metering_2  Sub_metering_3  Date_1/1/2008  \
0             0.0             1.0            17.0          False   
1             0.0             1.0            16.0          False   
2             0.0             2.0            17.0          False   
3             0.0             1.0            17.0          False   
4             0.0             1.0            17.0          False   

   Date_1/1/2009  Date_1/1/2010  ...  Time_23:50:00  Time_23:51:00  \
0          False          False  ...          False          Fal